# 13 — Interior analysis: daylight factor

Two models look *inside* a building rather than at a site:

| Model | Answers | Time input |
| --- | --- | --- |
| `daylight-factor` | daylight at each point in a room, as a % of unobstructed outdoor illuminance | none — the CIE overcast sky is time-independent |

**They are not area models.** You supply the room geometry, so there is no polygon, no
tiling and no merge — submit through `client.analyses.execute()`. `run_area()` rejects them.

The one thing to get right is the **entity shape**, and it is the one thing that fails
quietly. This notebook shows the trap first, then runs both models.


In [1]:
from dotenv import load_dotenv

load_dotenv()  # loads INFRARED_API_KEY from .env in this directory

import os

from infrared_sdk import (
    AnalysesName,
    DaylightFactorModelRequest,
    InfraredClient,
    interior_entities,  # noqa: F401  # used in the context-geometry section
    to_interior_entity,
)

client = InfraredClient(
    api_key=os.environ["INFRARED_API_KEY"],
    base_url=os.getenv("INFRARED_BASE_URL", "https://api-test.infrared.city"),
)
print("client ready")

client ready


## 1. The entity shape — and why the SDK is strict about it

The six outdoor grid models take **flat** meshes. Interior entities are **nested**:

```jsonc
{"geometry": {"payload": {"coordinates": [...], "indices": [...]}}}   // interior
{"coordinates": [...], "indices": [...]}                                // outdoor
```

Passing the flat shape is **not** a server-side error. The entity is read as having no
geometry, skipped, and the analysis runs against an empty occluder — so you get a
confident, near-uniform, physically meaningless field back with a 200.

The SDK refuses it locally instead.


In [2]:
def quad(p0, p1, p2, p3):
    """A flat two-triangle mesh — the shape the OUTDOOR models use."""
    return {
        "coordinates": [c for p in (p0, p1, p2, p3) for c in p],
        "indices": [0, 1, 2, 0, 2, 3],
    }


flat_floor = quad((0, 0, 0), (6, 0, 0), (6, 6, 0), (0, 6, 0))

try:
    DaylightFactorModelRequest(
        analysis_type=AnalysesName.daylight_factor,
        barriers={"floor": flat_floor},  # flat -> rejected
        sensor_points=[[3.0, 3.0, 0.8]],
    )
except ValueError as exc:
    print("rejected, as it should be:\n")
    print(str(exc).split("Value error, ")[-1][:400])

rejected, as it should be:

Invalid interior geometry:
  - barriers['floor']: flat mesh shape {'coordinates', 'indices'} — interior entities need {'geometry': {'payload': {...}}}. The server does NOT reject the flat shape; it reads an empty occluder and returns a non-physical result. [type=value_error, input_value={'analysis_type': <Analys...nts': [[3.0, 3.0, 0.8]]}, input_type=dict]
    For further information visit https:/


`to_interior_entity()` does the conversion. It accepts a flat dict, a `DotBimMesh`, or
an already-nested entity, so it is safe to call on anything.


In [3]:
floor = to_interior_entity(flat_floor)
print(floor)

{'geometry': {'payload': {'coordinates': [0, 0, 0, 6, 0, 0, 6, 6, 0, 0, 6, 0], 'indices': [0, 1, 2, 0, 2, 3]}}}


## 2. Build a room

A 6 × 6 × 3 m room: floor, ceiling, four walls, and one 2 × 2 m window in the south wall.

`openingFactor` is the glazing's visible-light transmittance. It is **per opening**, so
different windows can have different glazing — unlike `room_reflectances`, which is one
value for the whole request.


In [4]:
W = D = 6.0
H = 3.0

barriers = {
    "floor": to_interior_entity(quad((0, 0, 0), (W, 0, 0), (W, D, 0), (0, D, 0))),
    "ceiling": to_interior_entity(quad((0, 0, H), (W, 0, H), (W, D, H), (0, D, H))),
    "wall-south": to_interior_entity(quad((0, 0, 0), (W, 0, 0), (W, 0, H), (0, 0, H))),
    "wall-north": to_interior_entity(quad((0, D, 0), (W, D, 0), (W, D, H), (0, D, H))),
    "wall-west": to_interior_entity(quad((0, 0, 0), (0, D, 0), (0, D, H), (0, 0, H))),
    "wall-east": to_interior_entity(quad((W, 0, 0), (W, D, 0), (W, D, H), (W, 0, H))),
}

# 2 x 2 m window, centred in the south wall, sill at 0.9 m. Nudged just inside the
# wall plane so it is unambiguously an aperture in it.
EPS = 0.01
window = to_interior_entity(
    quad((2, EPS, 0.9), (4, EPS, 0.9), (4, EPS, 2.9), (2, EPS, 2.9)),
    opening_factor=0.7,
)
openings = {"window-south": window}

# Working-plane sensor grid at 0.8 m, 1 m pitch, on cell centres.
sensors = [[x + 0.5, y + 0.5, 0.8] for x in range(6) for y in range(6)]
print(f"{len(barriers)} barriers, {len(openings)} opening, {len(sensors)} sensors")

6 barriers, 1 opening, 36 sensors


## 3. Run the daylight factor


In [5]:
request = DaylightFactorModelRequest(
    analysis_type=AnalysesName.daylight_factor,
    barriers=barriers,
    openings=openings,
    sensor_points=sensors,
)

job = client.analyses.execute(payload=request)
client.jobs.wait_for_completion(job.job_id)

from infrared_sdk.analyses.jobs import JobsServiceClient

download = client.jobs.download_results(job.job_id)
result = JobsServiceClient.decompress(download.content)

values = [p["df"] for p in result["output"]]
print(f"{len(values)} sensors")
print(
    f"daylight factor  min {min(values):.2f}%  mean {sum(values) / len(values):.2f}%  max {max(values):.2f}%"
)

# A uniform field is the signature of a stripped occluder — worth asserting on.
assert len(set(values)) > 1, "uniform field: the occluder was empty"

[INFO] [SDK:Submit] analysis=daylight-factor url=https://api-test.infrared.city/async/daylight-factor raw_json_bytes=1690 zip_bytes=468


[INFO] [SDK:Submit] analysis=daylight-factor ok job_id=8ba25aeb-b095-47ab-b372-3e10be227cf6 elapsed_ms=406


36 sensors
daylight factor  min 2.37%  mean 4.66%  max 20.86%


## 4. Surrounding buildings

`context_geometry` is how neighbours shade the room. Omitting it does not error — the
room simply reads brighter than it is.

You can fetch real neighbours with the same call the outdoor models use, but note that
`get_area()` returns the **flat** shape, so it must be converted:

```python
area = client.buildings.get_area(site_polygon)
context = interior_entities(area)   # flat -> nested; required
```

Here we use three synthetic blocks to keep the notebook self-contained.


In [6]:
def block(x0, y0, x1, y1, height):
    """Four opaque walls standing on the ground — a crude massing block."""
    faces = [
        quad((x0, y0, 0), (x1, y0, 0), (x1, y0, height), (x0, y0, height)),
        quad((x0, y1, 0), (x1, y1, 0), (x1, y1, height), (x0, y1, height)),
        quad((x0, y0, 0), (x0, y1, 0), (x0, y1, height), (x0, y0, height)),
        quad((x1, y0, 0), (x1, y1, 0), (x1, y1, height), (x1, y0, height)),
    ]
    return {f"face-{i}": to_interior_entity(f) for i, f in enumerate(faces)}


context = {}
for name, coords in {
    "south": (-4, -14, 10, -8),
    "east": (12, -4, 24, 10),
    "west": (-16, -4, -4, 10),
}.items():
    for key, entity in block(*coords, height=18.0).items():
        context[f"{name}-{key}"] = entity

shaded = client.analyses.execute(
    payload=DaylightFactorModelRequest(
        analysis_type=AnalysesName.daylight_factor,
        barriers=barriers,
        openings=openings,
        sensor_points=sensors,
        context_geometry=context,
    )
)
client.jobs.wait_for_completion(shaded.job_id)
shaded_values = [
    p["df"]
    for p in JobsServiceClient.decompress(
        client.jobs.download_results(shaded.job_id).content
    )["output"]
]

bare_mean = sum(values) / len(values)
shaded_mean = sum(shaded_values) / len(shaded_values)
print(f"mean DF without context  {bare_mean:.2f}%")
print(f"mean DF with context     {shaded_mean:.2f}%")
print(f"reduction                {100 * (1 - shaded_mean / bare_mean):.0f}%")

[INFO] [SDK:Submit] analysis=daylight-factor url=https://api-test.infrared.city/async/daylight-factor raw_json_bytes=3436 zip_bytes=661


[INFO] [SDK:Submit] analysis=daylight-factor ok job_id=9bdbc4a9-c125-40b6-a46e-735c28887595 elapsed_ms=163


mean DF without context  4.66%
mean DF with context     2.82%
reduction                40%


## 5. Guard rails worth knowing

All of these are checked locally, before anything is submitted — because a
malformed request is **billed and then refused**: the token deduction happens
before the model runs.

| You do this | What the server would do | What the SDK does |
| --- | --- | --- |
| flat entity shape | silently skips it → empty occluder | rejects, naming the entity |
| room mesh in `geometries` | never reads it → uniform field | rejects, points at `barriers` |
| nested mesh in `ground_geometry` | 500, and the job is billed | rejects, names the flat shape |
| `opening_factor` (snake) | never reads it → glazing defaults to 1.0 | rejects, gives the spelling |
| `openingFactor=7.5` | no clamp → DF above 100% | rejects, bounds it to [0, 1] |
| non-finite / empty / out-of-range mesh | opaque 500, silent drop, or billed 500 | rejects locally |

Two fields are the exception to the nested rule: **`ground_geometry` and `vegetation`
take the flat shape**. One payload, two conventions — that is the server contract.

> ⚠️ **Request-size caps are NOT checked by the SDK.** The server caps a request at
> 15 floors (summed across all buildings), 100,000 sensors per floor and 2,000,000
> occluder triangles — and an over-cap request is accepted, **billed**, and only then
> refused. Count before you submit; client-side enforcement is not in this release.


In [7]:
# The guards above are client-side and cost nothing — each raises before
# any request is sent.
from infrared_sdk import AnalysesName, DaylightFactorModelRequest

for label, bad in [
    (
        "flat entity shape",
        {"barriers": {"floor": quad((0, 0, 0), (W, 0, 0), (W, D, 0), (0, D, 0))}},
    ),
    # The realistic case: only the snake spelling is set, so the server would
    # never read it and the window would silently be a clear pane.
    (
        "glazing misspelled",
        {
            "barriers": barriers,
            "openings": {"w": {"geometry": window["geometry"], "opening_factor": 0.1}},
        },
    ),
    (
        "glazing out of range",
        {
            "barriers": barriers,
            "openings": {"w": {"geometry": window["geometry"], "openingFactor": 7.5}},
        },
    ),
]:
    try:
        DaylightFactorModelRequest(
            analysis_type=AnalysesName.daylight_factor, sensor_points=sensors, **bad
        )
        print(f"{label:22s} -> ACCEPTED (unexpected)")
    except ValueError:
        print(f"{label:22s} -> rejected before submitting")

flat entity shape      -> rejected before submitting
glazing misspelled     -> rejected before submitting
glazing out of range   -> rejected before submitting
